# Meter API — Basic Query Demo

This notebook demonstrates how to authenticate and perform simple queries against the **Meter GraphQL API**.

## Authentication

The Meter API uses **Bearer token authentication**. Every request must include an `Authorization` header:

```
Authorization: Bearer YOUR_API_KEY
```

API keys are created in the Dashboard under **Settings → Integrations → API keys**. Keys are scoped to the company where they were created and are displayed only once. Each key can be revoked instantly.

## Request Format

All API calls are HTTP POST requests to a single endpoint:

```
https://api.meter.com/api/v1/graphql
```

The request body is a JSON object with a single `query` field:

```json
{ "query": "{ companyBySlug(slug: \"acme\") { name } }" }
```

## Response Format

Successful responses return **HTTP 200** with a JSON body. Only the fields you requested are returned:

```json
{ "data": { "companyBySlug": { "name": "Acme Corp" } } }
```

GraphQL errors appear inside an `errors` array (sometimes alongside HTTP 200) with an `extensions.code` field.

## Setup — Imports and Configuration

Credentials are loaded from `config.py`. Edit that file to set your `API_TOKEN`, `COMPANY_SLUG`, and `NETWORK_UUID`.

In [ ]:
import json
import requests
import config

API_URL      = config.API_URL
API_TOKEN    = config.API_TOKEN
COMPANY_SLUG = config.COMPANY_SLUG
NETWORK_UUID = config.NETWORK_UUID
SERIAL = config.SERIAL

print(f"Endpoint : {API_URL}")
print(f"Company  : {COMPANY_SLUG}")
print(f"Network  : {NETWORK_UUID}")

## Authentication Helper

Every request needs two headers:
- `Content-Type: application/json` — the body is JSON-encoded GraphQL
- `Authorization: Bearer <token>` — your API key exactly as shown in the Dashboard

In [ ]:
def make_headers(token: str) -> dict:
    """
    Build the HTTP headers required for every Meter API request.

    Args:
        token: Your Meter API key (the full Bearer token string).

    Returns:
        Dict containing Content-Type and Authorization headers.
    """
    return {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}",
    }

# Verify the header format
sample_headers = make_headers(API_TOKEN)
print("Content-Type :", sample_headers["Content-Type"])
print("Authorization:", sample_headers["Authorization"][:30] + "...")

## Core Request Function

`run_query` sends any GraphQL query to the Meter API and returns the parsed JSON response.

**Rate-limit headers** are present on every response:
- `X-RateLimit-Remaining` — requests left in the current 60-second window
- `X-RateLimit-Reset` — RFC 1123 timestamp when the window resets (500 req/min limit)

In [ ]:
def run_query(query: str, token: str = API_TOKEN, url: str = API_URL) -> dict:
    """
    Execute a GraphQL query against the Meter API.

    Args:
        query: A valid GraphQL query string.
        token: Bearer token. Defaults to config.API_TOKEN.
        url:   Meter GraphQL endpoint. Defaults to config.API_URL.

    Returns:
        Parsed JSON response dict with 'data' on success, 'errors' on failure.

    Raises:
        requests.HTTPError: On 4xx/5xx responses (401, 400, 422, 429).
        requests.Timeout:   If no response within 60 seconds.
    """
    headers = make_headers(token)
    payload = {"query": query}
    response = requests.post(url, headers=headers, json=payload, timeout=60)
    response.raise_for_status()
    return response.json()


def print_rate_limit_info(response_headers: dict) -> None:
    """Display the rate-limit headers from an API response."""
    remaining = response_headers.get("X-RateLimit-Remaining", "N/A")
    reset_time = response_headers.get("X-RateLimit-Reset", "N/A")
    print(f"  Rate limit remaining : {remaining} requests")
    print(f"  Rate limit resets at : {reset_time}")

## Step 1 — Verify Authentication (`companyBySlug`)

The simplest way to verify your token is valid is to query `companyBySlug`.
- **HTTP 401** → your `API_TOKEN` in `config.py` is wrong or revoked
- **HTTP 200** with `data.companyBySlug` → authenticated successfully

In [ ]:
def get_company(slug: str) -> dict:
    """
    Fetch basic company information by its slug identifier.

    GraphQL query: companyBySlug(slug: String!) -> Company!

    Returns: uuid, slug, name, isCustomer, websiteDomain
    """
    query = f"""
    {{
      companyBySlug(slug: "{slug}") {{
        uuid
        slug
        name
        isCustomer
        websiteDomain
      }}
    }}
    """
    return run_query(query)


# --- Run it ---
try:
    result = get_company(COMPANY_SLUG)
    company = result.get("data", {}).get("companyBySlug", {})
    print("✓ Authenticated successfully")
    print(f"  Company name : {company.get('name')}")
    print(f"  UUID         : {company.get('uuid')}")
    print(f"  Is customer  : {company.get('isCustomer')}")
    print(f"  Website      : {company.get('websiteDomain')}")
except requests.HTTPError as e:
    print(f"✗ HTTP {e.response.status_code} — check your API_TOKEN in config.py")

## Step 2 — List Networks (`networksForCompany`)

Discovers all networks associated with a company. The `UUID` field from each network is used as `networkUUID` in most other queries.

In [ ]:
def get_networks(company_slug: str) -> dict:
    """
    List all networks associated with a company.

    GraphQL query: networksForCompany(companySlug: String!) -> [Network!]!

    Returns: UUID, label, slug for each network.
    """
    query = f"""
    {{
      networksForCompany(companySlug: "{company_slug}") {{
        UUID
        label
        slug
      }}
    }}
    """
    return run_query(query)


# --- Run it ---
result = get_networks(COMPANY_SLUG)
networks = result.get("data", {}).get("networksForCompany", [])
print(f"Found {len(networks)} network(s):")
for net in networks:
    print(f"  • {net.get('label'):<30} UUID: {net.get('UUID')}  slug: {net.get('slug')}")

## Step 3 — List Virtual Devices (`virtualDevicesForNetwork`)

Lists all logical devices (switches, access points, controllers) on a network.

A **VirtualDevice** is the logical representation of a physical device. It pairs with a **HardwareDevice** that holds the serial number and connectivity state.

Device types: `ACCESS_POINT`, `SWITCH`, `CONTROLLER`, `CELLULAR_GATEWAY`, `POWER_DISTRIBUTION_UNIT`, `OBSERVER`

In [ ]:
def get_virtual_devices(network_uuid: str) -> dict:
    """
    List all virtual devices (switches, APs, controllers) on a network.

    GraphQL query: virtualDevicesForNetwork(networkUUID: UUID!) -> [VirtualDevice!]!

    Returns: UUID, label, deviceType, deviceModel, isOnline
    """
    query = f"""
    {{
      virtualDevicesForNetwork(networkUUID: "{network_uuid}") {{
        UUID
        label
        deviceType
        deviceModel
        isOnline
      }}
    }}
    """
    return run_query(query)


# --- Run it ---
result = get_virtual_devices(NETWORK_UUID)
devices = result.get("data", {}).get("virtualDevicesForNetwork", [])
print(f"Found {len(devices)} device(s) on network {NETWORK_UUID}:")
for dev in devices:
    status = "online" if dev.get("isOnline") else "offline"
    print(f"  [{status:>7}] {dev.get('label'):<25} {dev.get('deviceType'):<20} {dev.get('deviceModel')}")

## Step 4 — List Active Clients (`networkClients`)

Retrieves all currently active clients connected to a network (both wired and wireless).

Each `NetworkClient` includes:
- `macAddress` — hardware MAC address
- `ip` — assigned IP address
- `isWireless` — `true` for Wi-Fi, `false` for wired
- `signal` — Wi-Fi signal strength in dBm (null for wired)
- `connectedVLAN` — VLAN assignment
- `connectedSSID` — SSID for wireless clients

In [ ]:
def get_network_clients(network_uuid: str) -> dict:
    """
    Retrieve all currently active clients connected to a network.

    GraphQL query: networkClients(networkUUID: UUID!) -> [NetworkClient!]!
    """
    query = f"""
    {{
      networkClients(networkUUID: "{network_uuid}") {{
        macAddress
        ip
        clientName
        isWireless
        signal
        lastSeen
        connectedVLAN {{
          name
          vlanID
        }}
        connectedSSID {{
          ssid
        }}
      }}
    }}
    """
    return run_query(query)


# --- Run it ---
result = get_network_clients(NETWORK_UUID)
clients = result.get("data", {}).get("networkClients", [])
wireless = [c for c in clients if c.get("isWireless")]
wired    = [c for c in clients if not c.get("isWireless")]

print(f"Total clients  : {len(clients)}")
print(f"  Wireless     : {len(wireless)}")
print(f"  Wired        : {len(wired)}")

if clients:
    print(f"\nFirst 5 clients:")
    for client in clients[:5]:
        conn_type = "Wi-Fi" if client.get("isWireless") else "Wired"
        name = client.get("clientName") or client.get("macAddress")
        ip   = client.get("ip") or "—"
        print(f"  • {name:<30} {ip:<18} {conn_type}")

## Step 5 — Hardware Device Lookup (`hardwareDevice`)

Looks up a specific physical hardware device by its serial number.

**HardwareDevice** represents the physical unit — serial number, model, and real-time backend connectivity status. Use `virtualDeviceUUID` to link to its logical counterpart for configuration queries.

In [ ]:
def get_hardware_device(serial_number: str) -> dict:
    """
    Look up a specific physical hardware device by its serial number.

    GraphQL query: hardwareDevice(serialNumber: String!) -> HardwareDevice!

    Returns: serialNumber, deviceType, deviceModel, isConnectedToBackend,
             macAddress, networkUUID, virtualDeviceUUID
    """
    query = f"""
    {{
      hardwareDevice(serialNumber: "{serial_number}") {{
        serialNumber
        deviceType
        deviceModel
        isConnectedToBackend
        macAddress
        networkUUID
        virtualDeviceUUID
      }}
    }}
    """
    return run_query(query)


# --- Device Serial ---
result = get_hardware_device(config.SERIAL)
device = result.get("data", {}).get("hardwareDevice", {})
print(json.dumps(device, indent=2))

## Step 6 — Full Raw Response

Shows the complete, unprocessed JSON response from the API — useful for exploring new fields or debugging unexpected results.

In [ ]:
# Full raw JSON response from companyBySlug
raw = get_company(COMPANY_SLUG)
print(json.dumps(raw, indent=2, default=str))

## Summary

| Query | Field | Description |
|---|---|---|
| `companyBySlug` | slug | Verify auth, get company UUID |
| `networksForCompany` | companySlug | List all networks |
| `virtualDevicesForNetwork` | networkUUID | List switches, APs, controllers |
| `networkClients` | networkUUID | List active wired + wireless clients |
| `hardwareDevice` | serialNumber | Look up physical device by serial |

All requests are HTTP POST to `https://api.meter.com/api/v1/graphql` with `Authorization: Bearer <token>`.

Always check `response["errors"]` even when HTTP status is 200 — GraphQL errors can be embedded in successful HTTP responses.